# Modern Python Handbook Supplement — additional concepts

> **Explain it like I am five:** These tools are different drawers in a toolbox. You do not use every tool for every job; you choose the one that makes the job simpler and clearer.

Learn the core syntax first. Use these modern features when they improve readability—not merely because they look clever.

## What this notebook covers

- type hints and dataclasses;
- unpacking, comprehensions, `enumerate`, and `zip`;
- f-strings, the walrus operator, and `match`;
- dictionary merging and useful collection types;
- context managers and `pathlib`;
- scope, `global`, `nonlocal`, and the main guard;
- practical helpers from `itertools` and `functools`.

The original handbook examples are preserved and expanded throughout.


## 1. Type hints: labels for humans and tools

Type hints are like labels on toy boxes: “cars go here” and “blocks go there.” They help readers, editors, and type checkers understand the intended values.

Python does **not** normally enforce type hints at runtime by itself.

```python
def greeting(name: str) -> str:
    return f"Hello, {name}"

age: int = 25
```

Modern Python can annotate collections with `list[int]`, `tuple[str, int]`, and `dict[str, int]`. Use `str | int` when either type is accepted, and `T | None` when a value may be missing.


In [1]:
def greeting(name: str) -> str:
    return f"Hello, {name}"

scores: dict[str, int] = {"Maya": 95}
print(greeting("Maya"))
print(scores)


Hello, Maya
{'Maya': 95}


In [2]:
def average(values: list[float]) -> float:
    """Return the average, or 0.0 for an empty list."""
    return sum(values) / len(values) if values else 0.0


def find_score(name: str, scores: dict[str, int]) -> int | None:
    return scores.get(name)


print(average([10.0, 20.0, 30.0]))
print(find_score("Maya", scores))
print(find_score("Noor", scores))


20.0
95
None


### Type-hint reminders

- Hints describe an agreement; they do not sanitize user input.
- Choose precise hints, but do not make simple code unreadable.
- Tools such as Pyright or mypy can check hints before runtime.
- `Any` means “skip useful type checking here,” so use it sparingly.


## 2. Dataclasses: less repeated class code

A dataclass is useful for an object whose main job is to hold related data. Python can generate `__init__`, `__repr__`, and equality behavior for us.


In [3]:
from dataclasses import dataclass, field

@dataclass
class Student:
    name: str
    score: int
    skills: list[str] = field(default_factory=list)

    def passed(self) -> bool:
        return self.score >= 40


student = Student("Maya", 95, ["Python", "SQL"])
print(student)
print("Passed:", student.passed())


Student(name='Maya', score=95, skills=['Python', 'SQL'])
Passed: True


Use `field(default_factory=list)` instead of `skills=[]`. A mutable default list could otherwise be accidentally shared between objects.


## 3. Unpacking and starred expressions

Unpacking opens a container and assigns its pieces to names. The star collects “everything else.”


In [4]:
first, second = (10, 20)
head, *middle, tail = [1, 2, 3, 4, 5]

print(first, second)
print("head:", head)
print("middle:", middle)
print("tail:", tail)

left = [1, 2]
right = [3, 4]
combined = [*left, *right]
print(combined)


10 20
head: 1
middle: [2, 3, 4]
tail: 5
[1, 2, 3, 4]


Function-call unpacking uses `*` for positional values and `**` for keyword values.


In [5]:
def describe_person(name, age, city):
    return f"{name} is {age} and lives in {city}."


positional = ["Asha", 24]
keywords = {"city": "Pune"}
print(describe_person(*positional, **keywords))


Asha is 24 and lives in Pune.


## 4. Comprehensions: transform a collection clearly

A comprehension is a compact loop used to build a collection.

Read this left to right: “make `value * value` for each `value` in `values` if the value is even.”


In [6]:
values = [1, 7, 12, 11, 22]
larger = [value for value in values if value > 8]
even_squares = [value * value for value in values if value % 2 == 0]

print(larger)
print(even_squares)


[12, 11, 22]
[144, 484]


Collection forms:

- list: `[expression for item in items]`
- set: `{expression for item in items}`
- dictionary: `{key: value for item in items}`
- generator: `(expression for item in items)`

Keep comprehensions short. If you need several conditions, side effects, or nested loops, a normal loop is usually easier to understand.


In [7]:
words = ["Python", "is", "clear", "and", "powerful"]

unique_lengths = {len(word) for word in words}
length_by_word = {word: len(word) for word in words}
lazy_lengths = (len(word) for word in words)

print(unique_lengths)
print(length_by_word)
print(list(lazy_lengths))


{2, 3, 5, 6, 8}
{'Python': 6, 'is': 2, 'clear': 5, 'and': 3, 'powerful': 8}
[6, 2, 5, 3, 8]


## 5. `enumerate`, `zip`, `any`, and `all`

- `enumerate` gives position-value pairs.
- `zip` walks through inputs together and stops at the shortest.
- `any` asks “is at least one item truthy?”
- `all` asks “is every item truthy?”


In [8]:
values = [1, 7, 12, 11, 22]

for index, value in enumerate(values, start=1):
    print(index, value)

names = ["Maya", "Noor", "Ravi"]
scores = [95, 72, 38]
print(list(zip(names, scores)))
print("Anyone passed?", any(score >= 40 for score in scores))
print("Everyone passed?", all(score >= 40 for score in scores))


1 1
2 7
3 12
4 11
5 22
[('Maya', 95), ('Noor', 72), ('Ravi', 38)]
Anyone passed? True
Everyone passed? False


## 6. F-strings: readable text formatting

Put expressions inside `{}`. Format specifications come after `:`.


In [9]:
product = "Notebook"
price = 49.5
discount = 0.15

print(f"{product} costs ₹{price:.2f}.")
print(f"Discount: {discount:.0%}")
print(f"Final price: ₹{price * (1 - discount):,.2f}")


Notebook costs ₹49.50.
Discount: 15%
Final price: ₹42.07


Useful specifications:

- `.2f` → two digits after the decimal;
- `,` → thousands separator;
- `.0%` → percentage with no decimal places;
- `>10`, `<10`, `^10` → right, left, or center alignment in width 10.


## 7. Walrus operator `:=`: assign inside an expression

The walrus both saves a value and uses it in the same expression. Use it only when it removes meaningful repetition and remains easy to read.

Original example:


In [10]:
if (length := len("Python")) > 3:
    print("length:", length)


length: 6


In [11]:
# Without repeating input.strip()
raw_inputs = ["  apple  ", "   ", " banana "]

cleaned = [text for raw in raw_inputs if (text := raw.strip())]
print(cleaned)


['apple', 'banana']


Parentheses often make a walrus expression clearer. Avoid squeezing several assignments into one clever line.


## 8. Structural pattern matching with `match`

`match` compares a value with patterns. `_` is the fallback or “anything else” case. It is useful when the shape of data matters, not only for replacing every `if` statement.

Original status example:


In [12]:
def status_message(status):
    match status:
        case 200:
            return "OK"
        case 404:
            return "Not Found"
        case _:
            return "Unknown"

print(status_message(404))


Not Found


Patterns can unpack sequences and mappings:


In [13]:
def describe_command(command):
    match command:
        case ["move", x, y]:
            return f"Move to ({x}, {y})"
        case ["say", *words]:
            return " ".join(words)
        case {"action": "quit"}:
            return "Closing"
        case _:
            return "Unknown command"


print(describe_command(["move", 10, 20]))
print(describe_command(["say", "hello", "world"]))
print(describe_command({"action": "quit", "reason": "done"}))


Move to (10, 20)
hello world
Closing


## 9. Dictionary merge and update

`left | right` creates a new dictionary. If a key appears in both, the **right-hand value wins**. `left |= right` changes `left` in place.


In [14]:
defaults = {"theme": "light", "font_size": 12}
user_choices = {"theme": "dark", "language": "en"}

merged = defaults | user_choices
print(merged)
print("defaults unchanged:", defaults)


{'theme': 'dark', 'font_size': 12, 'language': 'en'}
defaults unchanged: {'theme': 'light', 'font_size': 12}


## 10. Useful collection types

The `collections` module contains specialized containers:

- `Counter` counts repeated values;
- `defaultdict` creates a default value for missing keys;
- `deque` efficiently adds/removes items at both ends.


In [15]:
from collections import Counter, defaultdict, deque

votes = Counter(["red", "blue", "red", "green", "red", "blue"])
print("counts:", votes)
print("most common:", votes.most_common(2))

students_by_grade = defaultdict(list)
students_by_grade["A"].append("Maya")
students_by_grade["B"].append("Noor")
print(dict(students_by_grade))

queue = deque(["first", "second"])
queue.append("third")
print("served:", queue.popleft())
print("remaining:", queue)


counts: Counter({'red': 3, 'blue': 2, 'green': 1})
most common: [('red', 3), ('blue', 2)]
{'A': ['Maya'], 'B': ['Noor']}
served: first
remaining: deque(['second', 'third'])


## 11. Context managers: set up, use, clean up

A context manager is like borrowing a library book with an automatic return desk. The `with` statement performs cleanup even if an error occurs.

Files are the most common example:

```python
with open("notes.txt", "r", encoding="utf-8") as file:
    text = file.read()
# file is closed here
```

You can manage multiple resources in one `with` statement.


In [16]:
from contextlib import contextmanager

@contextmanager
def friendly_resource(name):
    print(f"Opening {name}")
    try:
        yield {"name": name}
    finally:
        print(f"Closing {name}")


with friendly_resource("demo") as resource:
    print("Using", resource["name"])


Opening demo
Using demo
Closing demo


The `finally` block is essential: cleanup runs whether the `with` block succeeds or fails.


## 12. `pathlib`: paths as objects

`pathlib.Path` provides readable, cross-platform file-path operations. `/` joins path pieces; it is not division here.


In [17]:
from pathlib import Path

course_folder = Path("Complete-Python-Bootcamp-main")
advanced_folder = course_folder / "9-Advance Python Concepts"

print("Path:", advanced_folder)
print("Exists:", advanced_folder.exists())
print("Notebook names:")
for path in sorted(advanced_folder.glob("*.ipynb")):
    print("-", path.name)


Path: Complete-Python-Bootcamp-main\9-Advance Python Concepts
Exists: True
Notebook names:
- 9.1-Iterators.ipynb
- 9.2-Generators.ipynb
- 9.3-Decorators.ipynb
- 9.4-handbook-modern-python-supplement.ipynb


Prefer an explicit `encoding="utf-8"` for text files. Before overwriting or deleting a path, verify that it points exactly where you expect.


## 13. Scope: local, enclosing, global, built-in

Python looks for a name using **LEGB**:

1. **Local** — current function;
2. **Enclosing** — outer function;
3. **Global** — module or notebook;
4. **Built-in** — names such as `len` and `print`.

Avoid `global` when possible: pass values into functions and return new values. `nonlocal` changes a name in the nearest enclosing function.


In [18]:
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment


counter = make_counter()
print(counter())
print(counter())
print(counter())


1
2
3


Use state like this deliberately. A small class can be clearer when several pieces of state or many operations belong together.


## 14. The main guard

When Python runs a file directly, `__name__` is `"__main__"`. When another file imports it, `__name__` is the module's name.

Put demo or command-line work behind this guard so imports do not run it automatically:

```python
def main():
    print("Running directly")

if __name__ == "__main__":
    main()
```

In a notebook, `__name__` is normally also `"__main__"`, so the demonstration runs.


In [19]:
def main():
    print("Running directly")


if __name__ == "__main__":
    main()


Running directly


## 15. `itertools`: iterator building blocks

`itertools` gives memory-friendly tools for combining, slicing, grouping, and accumulating iterables.


In [20]:
from itertools import accumulate, chain, islice

print("chain:", list(chain([1, 2], [3, 4])))
print("running totals:", list(accumulate([10, 20, 5])))
print("safe slice:", list(islice(range(1_000_000), 3)))


chain: [1, 2, 3, 4]
running totals: [10, 30, 35]
safe slice: [0, 1, 2]


`groupby` groups **consecutive** equal keys, so sort first when you want all matching items together.


In [21]:
from itertools import groupby

animals = ["ant", "ape", "bear", "bee", "cat"]
for first_letter, group in groupby(animals, key=lambda animal: animal[0]):
    print(first_letter, list(group))


a ['ant', 'ape']
b ['bear', 'bee']
c ['cat']


## 16. `functools`: reuse and caching

- `partial` pre-fills some function arguments.
- `lru_cache` remembers previous results for hashable arguments.
- `wraps` preserves metadata inside decorators.


In [22]:
from functools import lru_cache, partial

def power(base, exponent):
    return base ** exponent


square = partial(power, exponent=2)
print(square(9))

@lru_cache(maxsize=None)
def fibonacci(number):
    if number < 2:
        return number
    return fibonacci(number - 1) + fibonacci(number - 2)


print(fibonacci(30))
print(fibonacci.cache_info())


81
832040
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


Cache only functions whose result depends on their arguments and does not need to reflect changing outside state.


## 17. Python style: EAFP and clear code

Python often follows **EAFP**: “Easier to Ask Forgiveness than Permission.” Try the operation and catch the specific expected exception.


In [23]:
user = {"name": "Maya"}

try:
    city = user["city"]
except KeyError:
    city = "Unknown"

print(city)


Unknown


Do not use a bare `except:`. Catch the smallest specific exception you expect, and keep the `try` block narrow so unrelated bugs are not hidden.


## 18. Practice

1. Add type hints to a calculator.
2. Create a `Book` dataclass with `title`, `author`, and `pages`.
3. Rewrite a long exact-value `if`/`elif` function with `match`.
4. Use `enumerate` to print positions starting at 1.
5. Merge two dictionaries and explain which duplicate value wins.
6. Convert one simple loop into a readable comprehension.
7. Count words using `Counter`.
8. Explain why `with` is safer for cleanup.
9. Use `partial` to create a function that cubes a number.
10. Explain the difference between `global` and `nonlocal`.


In [24]:
# Sample answers for selected practice questions
from collections import Counter
from dataclasses import dataclass
from functools import partial

def calculator(first: float, second: float, operation: str) -> float:
    match operation:
        case "add":
            return first + second
        case "subtract":
            return first - second
        case _:
            raise ValueError("Unsupported operation")

@dataclass
class Book:
    title: str
    author: str
    pages: int

cube = partial(power, exponent=3)

print(calculator(8, 3, "subtract"))
print(Book("Learning Python", "A. Student", 350))
print(Counter("red blue red green red".split()))
print(cube(4))


5
Book(title='Learning Python', author='A. Student', pages=350)
Counter({'red': 3, 'blue': 1, 'green': 1})
64


## Easy revision cheat sheet

| Goal | Modern Python pattern |
|---|---|
| Document types | `def f(x: int) -> str:` |
| Optional value | `int | None` |
| Data-focused class | `@dataclass` |
| Collect leftovers | `first, *rest = values` |
| Build a list | `[f(x) for x in items if condition]` |
| Number items | `enumerate(items, start=1)` |
| Pair items | `zip(left, right)` |
| Test at least one | `any(condition for x in items)` |
| Test every one | `all(condition for x in items)` |
| Format text | `f"{value:.2f}"` |
| Assign and test once | `if (value := expression):` |
| Match shapes/cases | `match value: case pattern:` |
| Merge dictionaries | `merged = left | right` |
| Count values | `Counter(items)` |
| Automatic cleanup | `with resource as name:` |
| Work with paths | `Path(folder) / "file.txt"` |
| Change enclosing name | `nonlocal name` |
| Direct-run entry point | `if __name__ == "__main__":` |
| Join lazy inputs | `itertools.chain(...)` |
| Pre-fill arguments | `functools.partial(...)` |
| Cache pure results | `@lru_cache(...)` |

**Final memory trick:** Prefer the clearest ordinary code. Reach for a modern feature when it removes repetition, expresses the idea more directly, or prevents mistakes.
